# Stereo Line Reconstruction Demo

This notebook demonstrates **3D line reconstruction from a stereo image pair**
using the LineExtraction library. It covers the full pipeline:

1. Loading a calibrated stereo pair from the **Middlebury Stereo** (MDB) dataset
2. Detecting line segments with **LSD** (CC detector) in both views
3. Computing **LBD** descriptors from native Sobel gradients
4. **Stereo-filtered matching** — geometric epipolar constraint + descriptor similarity
5. **Triangulating** matched segments into 3D using three methods
6. **Reprojection error** analysis and filtering
7. **Merging** near-colinear 3D segments
8. Optional: point-feature baseline for pose verification
9. Interactive **Rerun** visualization — 2D matches + 3D reconstruction + camera frustums
10. Quantitative **evaluation** of reconstruction quality

## Prerequisites

```bash
# Build all Python bindings
bazel build //libs/...

# Install notebook deps (in the project .venv)
pip install rerun-sdk[notebook] Pillow

# Download MDB stereo dataset (required — downloads ~300 MB)
./tools/scripts/setup_mdb_dataset.sh
```

> Change `SCENE` and `RESOLUTION` in the configuration cell to experiment
> with different stereo pairs. Available scenes include Adirondack, Jadeplant,
> Motorcycle, Piano, Pipes, Playroom, Playtable, Recycle, Shelves, Vintage.

## 1. Import Required Libraries

Import all necessary modules: `le_geometry` (stereo triangulation, cameras),
`le_lsd` (line segment detection), `le_lfd` (descriptors, matchers, filters),
OpenCV for image I/O, NumPy for arrays, and Rerun for visualization.

In [ ]:
import sys
import pathlib
import time
import colorsys

import numpy as np
import cv2
import rerun as rr
import rerun.blueprint as rrb
from PIL import Image

# --- Discover workspace root and add binding paths ---
def _find_workspace_root() -> pathlib.Path:
    """Walk up from CWD until we find MODULE.bazel."""
    p = pathlib.Path.cwd()
    while p != p.parent:
        if (p / "MODULE.bazel").exists():
            return p
        p = p.parent
    raise RuntimeError("Cannot find workspace root (MODULE.bazel)")

ROOT = _find_workspace_root()
for lib in ["geometry", "imgproc", "edge", "lsd", "lfd", "algorithm", "eval"]:
    sp = str(ROOT / "bazel-bin" / "libs" / lib / "python")
    if sp not in sys.path:
        sys.path.insert(0, sp)
sp = str(ROOT / "python")
if sp not in sys.path:
    sys.path.insert(0, sp)

import le_geometry
import le_lsd
import le_lfd
from lsfm.data import TestImages

print(f"Workspace: {ROOT}")
print(f"le_geometry stereo: {[x for x in dir(le_geometry) if 'Stereo' in x]}")
print(f"le_geometry cameras: {[x for x in dir(le_geometry) if 'Camera' in x]}")
print(f"le_lfd matchers:     {[x for x in dir(le_lfd) if 'BruteForce' in x or 'Matcher' in x]}")
print(f"le_lfd filters:      {[x for x in dir(le_lfd) if 'Filter' in x]}")
print(f"Rerun version:       {rr.__version__}")

## 2. Load Stereo Image Pair and Detect Lines

We load a calibrated stereo pair from the **Middlebury MiddEval3** dataset.
The `TestImages` class provides:
- `stereo_pair(scene, resolution)` — paths to left/right images
- `stereo_calibration(scene, resolution)` — intrinsic matrices & baseline
- `stereo_camera_pair(scene, resolution)` — ready-to-use `Camera_f64` objects

Then we run `LsdCC` (connected-component LSD) on both images to detect
line segments. The detector also provides native Sobel gradients for
descriptor computation.

In [ ]:
# ================================================================
# Configuration — change these to experiment
# ================================================================
SCENE = "Motorcycle"         # MDB scene name (Motorcycle has strong line structure)
RESOLUTION = "H"             # "Q" (quarter), "H" (half), "F" (full) disparity range
DOWNSCALE = 0.5              # image downscale factor (0.5 = half size)
MIN_SEG_LENGTH = 20          # minimum segment length (pixels, after downscale)
LSD_TH_LOW = 0.004           # LSD lower gradient threshold (default 0.004)
LSD_TH_HIGH = 0.012          # LSD upper gradient threshold (default 0.012)
LSD_MIN_PIX = 10             # LSD minimum supporting pixels (default 10)
STEREO_ANGLE_TH = 10.0       # stereo filter angle threshold (degrees)
STEREO_OVERLAP = 0.25        # stereo filter minimum y-overlap ratio
MAX_DISP = 0                 # max disparity (0 = auto from ndisp*1.5)
N_SHOW = 200                 # number of matches to visualize
REPROJ_THRESHOLD = 10.0      # max reprojection error to keep (pixels)
MERGE_ANGLE_TH = 5.0         # colinear merge angle threshold (degrees)
MERGE_DIST_TH = 2.0          # colinear merge distance threshold (world units)
RATIO_THRESH = 0.80          # descriptor ratio test threshold
# Outlier detection: adaptive angle/length thresholds
OUTLIER_Z_ANGLE_HARD = 10.0  # hard limit: always reject if Z-angle < this (degrees)
OUTLIER_Z_ANGLE_SOFT = 30.0  # soft limit: short segments rejected up to this angle
OUTLIER_LENGTH_SAT = 500.0   # 3D length at which the angle threshold relaxes to hard limit
# Use the C++ StereoLineFilter during matching for efficient geometric pruning.
# The midpoint-based disparity fix ensures correct behavior with LSD segments.
USE_STEREO_MATCHER = False
# ================================================================

In [ ]:
images = TestImages()

# Load images
try:
    left_path, right_path = images.stereo_pair(SCENE, RESOLUTION)
    img_left_color = np.array(Image.open(left_path).convert("RGB"))
    img_right_color = np.array(Image.open(right_path).convert("RGB"))
except FileNotFoundError:
    raise FileNotFoundError(
        f"MDB dataset not found for '{SCENE}'. "
        f"Run: ./tools/scripts/setup_mdb_dataset.sh"
    )

# Downscale if requested
if DOWNSCALE != 1.0:
    new_w = int(img_left_color.shape[1] * DOWNSCALE)
    new_h = int(img_left_color.shape[0] * DOWNSCALE)
    img_left_color = cv2.resize(img_left_color, (new_w, new_h), interpolation=cv2.INTER_AREA)
    img_right_color = cv2.resize(img_right_color, (new_w, new_h), interpolation=cv2.INTER_AREA)

img_left = cv2.cvtColor(img_left_color, cv2.COLOR_RGB2GRAY)
img_right = cv2.cvtColor(img_right_color, cv2.COLOR_RGB2GRAY)
print(f"Scene: {SCENE} ({RESOLUTION}), downscale={DOWNSCALE}")
print(f"Left:  {img_left.shape[1]}x{img_left.shape[0]}")
print(f"Right: {img_right.shape[1]}x{img_right.shape[0]}")

# Load calibration and scale intrinsics for downscaled images
calib = images.stereo_calibration(SCENE, RESOLUTION)
if DOWNSCALE != 1.0:
    calib = dict(calib)  # copy so original is not mutated
    calib["focal_x"] *= DOWNSCALE
    calib["focal_y"] *= DOWNSCALE
    calib["offset_x"] *= DOWNSCALE
    calib["offset_y"] *= DOWNSCALE
    calib["baseline"] = calib.get("baseline", 0)  # baseline is metric, unchanged
    calib["doffs"] = calib.get("doffs", 0) * DOWNSCALE
    calib["ndisp"] = calib.get("ndisp", 0) * DOWNSCALE
    # Scale cam1 matrix if present
    if "cam1" in calib and calib["cam1"] is not None:
        cam1_scaled = [row[:] for row in calib["cam1"]]
        cam1_scaled[0][0] *= DOWNSCALE  # fx
        cam1_scaled[0][2] *= DOWNSCALE  # cx
        cam1_scaled[1][1] *= DOWNSCALE  # fy
        cam1_scaled[1][2] *= DOWNSCALE  # cy
        calib["cam1"] = cam1_scaled
    calib["width"] = img_left.shape[1]
    calib["height"] = img_left.shape[0]

# Auto-compute max disparity from calibration if not set
if MAX_DISP <= 0:
    MAX_DISP = int(calib.get("ndisp", 280) * 1.5)
    print(f"\nAuto max_disp: {MAX_DISP} (from ndisp={calib.get('ndisp', 280):.0f})")

# Build cameras (matching stereo_camera_pair logic)
from le_geometry import Camera_f64
fx_cal = calib["focal_x"]
fy_cal = calib["focal_y"]
cx_cal = calib["offset_x"]
cy_cal = calib["offset_y"]
doffs_cal = calib.get("doffs", 0)
baseline_cal = calib.get("baseline", 0)

cam_left = Camera_f64(
    focal_x=fx_cal, focal_y=fy_cal,
    offset_x=cx_cal, offset_y=cy_cal,
    width=img_left.shape[1], height=img_left.shape[0],
)

# Right camera: cx_right = cam1[0][2] if available, else cx + doffs
cam1_mat = calib.get("cam1")
if cam1_mat is not None:
    cx_right_cam = cam1_mat[0][2]
else:
    cx_right_cam = cx_cal + doffs_cal

cam_right = Camera_f64(
    focal_x=fx_cal, focal_y=fy_cal,
    offset_x=cx_right_cam, offset_y=cy_cal,
    width=img_right.shape[1], height=img_right.shape[0],
    tx=baseline_cal,
)

print(f"\nCalibration (scaled):")
print(f"  Focal: ({fx_cal:.1f}, {fy_cal:.1f})")
print(f"  Offset left: ({cx_cal:.1f}, {cy_cal:.1f})")
print(f"  Offset right: ({cx_right_cam:.1f}, {cy_cal:.1f})")
print(f"  Baseline: {baseline_cal:.2f}")
print(f"  doffs: {doffs_cal:.2f}")
print(f"  ndisp: {calib.get('ndisp', 0)}")

# Detect line segments in both images
t0 = time.perf_counter()
det_left = le_lsd.LsdCC(th_low=LSD_TH_LOW, th_high=LSD_TH_HIGH, min_pix=LSD_MIN_PIX)
det_left.detect(img_left)
seg_left_all = det_left.line_segments()

det_right = le_lsd.LsdCC(th_low=LSD_TH_LOW, th_high=LSD_TH_HIGH, min_pix=LSD_MIN_PIX)
det_right.detect(img_right)
seg_right_all = det_right.line_segments()
t_detect = time.perf_counter() - t0

# Filter by minimum length
seg_left = [s for s in seg_left_all if s.length >= MIN_SEG_LENGTH]
seg_right = [s for s in seg_right_all if s.length >= MIN_SEG_LENGTH]

print(f"\nDetection ({t_detect*1000:.1f} ms):")
print(f"  LSD params: th_low={LSD_TH_LOW}, th_high={LSD_TH_HIGH}, min_pix={LSD_MIN_PIX}")
print(f"  Left:  {len(seg_left_all)} total, {len(seg_left)} >= {MIN_SEG_LENGTH}px")
print(f"  Right: {len(seg_right_all)} total, {len(seg_right)} >= {MIN_SEG_LENGTH}px")

## 3. Extract Line Feature Descriptors (LBD)

We compute **LBD** (Line Band Descriptor) descriptors using the **native Sobel
gradients** from the LSD detector — these are `int16` images with typical values
$\pm$ hundreds, matching the intended input range of the C++ `FdcLBD` creator.

Each descriptor is a 72-dimensional float vector capturing the local gradient
structure around the line segment.

In [ ]:
t0 = time.perf_counter()

# Use native Sobel gradients from LSD (int16 → float32)
gx_left = det_left.image_data()[0].astype(np.float32)
gy_left = det_left.image_data()[1].astype(np.float32)
gx_right = det_right.image_data()[0].astype(np.float32)
gy_right = det_right.image_data()[1].astype(np.float32)

lbd_left = le_lfd.FdcLBD(gx_left, gy_left)
desc_left = lbd_left.create_list(seg_left)

lbd_right = le_lfd.FdcLBD(gx_right, gy_right)
desc_right = lbd_right.create_list(seg_right)

# Also get descriptor matrices for statistics
desc_mat_left = lbd_left.create_mat(seg_left)
desc_mat_right = lbd_right.create_mat(seg_right)

t_desc = time.perf_counter() - t0

print(f"LBD Descriptors ({t_desc*1000:.1f} ms):")
print(f"  Dimension: {lbd_left.size()}")
print(f"  Left:  {len(desc_left)} descriptors, shape {desc_mat_left.shape}")
print(f"  Right: {len(desc_right)} descriptors, shape {desc_mat_right.shape}")
print(f"  Gradient dtype: {gx_left.dtype}, range: [{gx_left.min():.0f}, {gx_left.max():.0f}]")

# Descriptor norm statistics
norms_left = np.linalg.norm(desc_mat_left, axis=1)
norms_right = np.linalg.norm(desc_mat_right, axis=1)
print(f"  L2 norm — left:  mean={norms_left.mean():.2f}, std={norms_left.std():.2f}")
print(f"  L2 norm — right: mean={norms_right.mean():.2f}, std={norms_right.std():.2f}")

## 4. Match Line Features Across Views

We use `BruteForceLBD` for descriptor-based matching. The raw best matches
give us putative correspondences ranked by L2 descriptor distance.

In [ ]:
t0 = time.perf_counter()

if USE_STEREO_MATCHER:
    # Integrated stereo matching: StereoLineFilter prunes geometrically
    # invalid candidates *during* distance computation — more efficient
    # than separate brute-force + post-filter.
    stereo_filter = le_lfd.StereoLineFilter(
        height=img_left.shape[0],
        max_dis_px=float(MAX_DISP),
        angle_th=float(STEREO_ANGLE_TH),
        min_y_overlap=float(STEREO_OVERLAP),
    )
    stereo_filter.train(seg_left, seg_right)

    matcher_raw = le_lfd.BruteForceLBD()
    matcher_raw.train_filtered_stereo(desc_left, desc_right, stereo_filter)
    raw_best = matcher_raw.best()
    raw_knn = matcher_raw.knn(2)
    match_method = "StereoLineFilter + BruteForceLBD"
else:
    # Plain brute-force matching (geometry filter applied separately)
    matcher_raw = le_lfd.BruteForceLBD()
    matcher_raw.train(desc_left, desc_right)
    raw_best = matcher_raw.best()
    raw_knn = matcher_raw.knn(2)
    match_method = "BruteForceLBD"

# Group knn results by query index — with filtered matching, not every
# query has 2 neighbors, so flat 2*i indexing is unsafe.
knn_by_query = {}
for dm in raw_knn:
    qid = dm.query_idx
    if qid not in knn_by_query:
        knn_by_query[qid] = []
    knn_by_query[qid].append(dm)

# Collect valid raw matches with ratio test
matches_raw = []
for i, m in enumerate(raw_best):
    if np.isnan(m.distance):
        continue
    # Ratio test: reject ambiguous matches
    neighbors = knn_by_query.get(i, [])
    if len(neighbors) >= 2:
        nn0, nn1 = neighbors[0], neighbors[1]
        if not np.isnan(nn1.distance) and nn1.distance > 0:
            if nn0.distance / nn1.distance >= RATIO_THRESH:
                continue
    matches_raw.append((i, m))

matches_raw.sort(key=lambda p: p[1].distance)
t_raw_match = time.perf_counter() - t0

print(f"Matching [{match_method}] ({t_raw_match*1000:.1f} ms):")
print(f"  Putative matches (ratio test {RATIO_THRESH}): {len(matches_raw)}")
if matches_raw:
    dists = [m.distance for _, m in matches_raw]
    print(f"  Distance range: [{dists[0]:.4f}, {dists[-1]:.4f}]")
    print(f"  Median distance: {np.median(dists):.4f}")

## 5. Filter Matches with Geometric Constraints

The `StereoLineFilter` enforces geometric constraints between stereo pairs:
- **Angle consistency** — matched segments must have similar orientations
- **Vertical overlap** — segments must overlap vertically (epipolar constraint)
- **Maximum disparity** — limits the horizontal displacement

We also apply `GlobalRotationFilter` for additional rotation-consistency
filtering. These dramatically reduce false matches compared to descriptor-only
matching.

In [ ]:
t0 = time.perf_counter()


# ---- Python stereo geometry filter ----
# When USE_STEREO_MATCHER=True, the C++ StereoLineFilter already applies
# midpoint-based disparity checks during matching. This Python filter serves
# as an additional safety net (e.g., stricter thresholds or different logic).
# When USE_STEREO_MATCHER=False, this is the primary geometry filter.

def _seg_angle(seg):
    """Compute angle of a segment in degrees [0, 180)."""
    x1, y1, x2, y2 = seg.end_points()
    a = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180.0
    return a

def _seg_y_range(seg):
    """Return (y_min, y_max) of a segment."""
    x1, y1, x2, y2 = seg.end_points()
    return (min(y1, y2), max(y1, y2))

def _midpoint_disparity(seg_l, seg_r):
    """Compute disparity at segment midpoints."""
    cl = seg_l.center_point()
    cr = seg_r.center_point()
    return cl[0] - cr[0]

def stereo_geometry_filter(seg_l, seg_r, max_disp, angle_th, min_overlap):
    """Check stereo geometric constraints for a segment pair.

    Returns True if the pair PASSES the filter (is a valid candidate).
    """
    # 1. Midpoint disparity: must be positive and within range
    disp = _midpoint_disparity(seg_l, seg_r)
    if disp <= 0 or disp > max_disp:
        return False

    # 2. Angle similarity
    a_l = _seg_angle(seg_l)
    a_r = _seg_angle(seg_r)
    adiff = abs(a_l - a_r)
    if adiff > 90:
        adiff = 180 - adiff
    if adiff > angle_th:
        return False

    # 3. Y-overlap: segments must share vertical extent
    y_min_l, y_max_l = _seg_y_range(seg_l)
    y_min_r, y_max_r = _seg_y_range(seg_r)
    overlap = min(y_max_l, y_max_r) - max(y_min_l, y_min_r)
    if overlap <= 0:
        return False
    len_l = y_max_l - y_min_l
    len_r = y_max_r - y_min_r
    min_len = min(len_l, len_r)
    if min_len > 0 and overlap / min_len < min_overlap:
        return False

    return True


# Apply geometric filter to raw descriptor matches
# Note: We skip GlobalRotationFilter — it's designed for arbitrary rotation
# between views and is too aggressive for rectified stereo pairs.
matches_filtered = []
for i, m in matches_raw:
    if stereo_geometry_filter(
        seg_left[i], seg_right[m.match_idx],
        MAX_DISP, STEREO_ANGLE_TH, STEREO_OVERLAP
    ):
        matches_filtered.append((i, m))

t_filter = time.perf_counter() - t0

print(f"Geometric Filtering ({t_filter*1000:.1f} ms):")
print(f"  Raw (ratio test):     {len(matches_raw)}")
print(f"  + Stereo geometry:    {len(matches_filtered)} (disp in (0, {MAX_DISP}], "
      f"angle<{STEREO_ANGLE_TH}°, y-overlap>{STEREO_OVERLAP})")
if matches_filtered:
    disps = [_midpoint_disparity(seg_left[i], seg_right[m.match_idx])
             for i, m in matches_filtered]
    print(f"    Disparity range: [{min(disps):.0f}, {max(disps):.0f}] px")
    print(f"    Mean descriptor dist: "
          f"{np.mean([m.distance for _, m in matches_filtered]):.4f}")

# Use geometry-verified matches
matches = matches_filtered

## 6. Estimate Camera Poses from Point Features (Baseline)

As a baseline check, we detect and match **SIFT point features** across the
stereo pair. Since MDB provides calibrated cameras, we verify that the
calibration is consistent with the point-feature essential matrix.

We construct projection matrices $P_1 = K[I|0]$ and $P_2 = K[R|t]$ where
$R = I$ and $t = (b, 0, 0)$ for a rectified stereo pair with baseline $b$.

In [ ]:
# Build projection matrices from (possibly scaled) MDB calibration
# Reuse calibration values already scaled in the config cell
fx = calib["focal_x"]
fy = calib["focal_y"]
cx = calib["offset_x"]
cy = calib["offset_y"]
baseline = calib.get("baseline", 0)
doffs = calib.get("doffs", 0)

K_left = np.array([[fx, 0, cx],
                    [0, fy, cy],
                    [0,  0,  1]], dtype=np.float64)

# Right camera: use already-computed cx_right_cam from config cell
cx_right = cx_right_cam  # already scaled if DOWNSCALE != 1
K_right = np.array([[fx, 0, cx_right],
                     [0, fy, cy],
                     [0,  0,  1]], dtype=np.float64)

P1 = K_left @ np.hstack([np.eye(3), np.zeros((3, 1))])
# P = K[I|t] where t = -camera_center; right camera center is at (baseline, 0, 0)
P2 = K_right @ np.hstack([np.eye(3), np.array([[-baseline], [0], [0]])])

print("Projection matrices from MDB calibration:")
print(f"  K_left:\n{K_left}")
print(f"  Baseline: {baseline:.2f}")
print(f"  P1 shape: {P1.shape}, P2 shape: {P2.shape}")

# Point feature baseline check: SIFT matching + triangulation
# NOTE: SIFT is run in a subprocess to avoid kernel crashes caused by
# symbol conflicts between the Bazel-built pybind11 bindings (linked
# against one OpenCV version) and the pip-installed opencv-python.
import subprocess, json, textwrap

_sift_script = textwrap.dedent(f"""\
import cv2, numpy as np, json, sys
from PIL import Image

img_l = np.array(Image.open("{left_path}").convert("L"))
img_r = np.array(Image.open("{right_path}").convert("L"))

# Downscale to match notebook resolution
ds = {DOWNSCALE}
if ds != 1.0:
    new_w = int(img_l.shape[1] * ds)
    new_h = int(img_l.shape[0] * ds)
    img_l = cv2.resize(img_l, (new_w, new_h), interpolation=cv2.INTER_AREA)
    img_r = cv2.resize(img_r, (new_w, new_h), interpolation=cv2.INTER_AREA)

K = np.array({K_left.tolist()}, dtype=np.float64)
P1 = np.array({P1.tolist()}, dtype=np.float64)
P2 = np.array({P2.tolist()}, dtype=np.float64)

sift = cv2.SIFT_create(nfeatures=2000)
kp1, des1 = sift.detectAndCompute(img_l, None)
kp2, des2 = sift.detectAndCompute(img_r, None)

bf = cv2.BFMatcher(cv2.NORM_L2)
pt_matches = bf.knnMatch(des1, des2, k=2)

good = [m for m, n in pt_matches if m.distance < 0.7 * n.distance]
pts1 = np.float64([kp1[m.queryIdx].pt for m in good])
pts2 = np.float64([kp2[m.trainIdx].pt for m in good])

E, mask_e = cv2.findEssentialMat(pts1, pts2, K, method=cv2.RANSAC,
                                  threshold=1.0, prob=0.999)
n_inliers = int(mask_e.sum()) if mask_e is not None else 0
_, R_est, t_est, _ = cv2.recoverPose(E, pts1, pts2, K, mask=mask_e)

rot_err = float(np.linalg.norm(R_est - np.eye(3), 'fro'))
t_dir = t_est.flatten() / (np.linalg.norm(t_est) + 1e-12)
t_err = float(np.degrees(np.arccos(np.clip(abs(t_dir[0]), 0, 1))))

# Triangulate inlier points using calibrated projection matrices
inlier_mask = mask_e.ravel().astype(bool) if mask_e is not None else np.ones(len(pts1), dtype=bool)
pts1_in = pts1[inlier_mask]
pts2_in = pts2[inlier_mask]
pts4d = cv2.triangulatePoints(P1, P2, pts1_in.T, pts2_in.T)
pts3d = (pts4d[:3] / pts4d[3:]).T  # (N, 3)

# Filter: positive depth, finite, reasonable range
valid = (pts3d[:, 2] > 0) & np.all(np.isfinite(pts3d), axis=1) & np.all(np.abs(pts3d) < 1e6, axis=1)
pts3d_valid = pts3d[valid]

json.dump(dict(
    n_kp1=len(kp1), n_kp2=len(kp2), n_good=len(good),
    n_inliers=n_inliers,
    R_est=R_est.tolist(), t_est=t_est.flatten().tolist(),
    rot_err=rot_err, t_err=t_err,
    pts3d=pts3d_valid.tolist(),
    n_pts3d_valid=len(pts3d_valid),
), sys.stdout)
""")

proc = subprocess.run(
    [sys.executable, "-c", _sift_script],
    capture_output=True, text=True, timeout=120,
)

sift_pts3d = np.empty((0, 3))
if proc.returncode == 0:
    r = json.loads(proc.stdout)
    R_est = np.array(r["R_est"])
    t_est = np.array(r["t_est"])
    sift_pts3d = np.array(r.get("pts3d", []))
    print(f"\nPoint feature verification:")
    print(f"  SIFT keypoints: {r['n_kp1']} left, {r['n_kp2']} right")
    print(f"  Good matches (ratio test): {r['n_good']}")
    print(f"  Essential matrix inliers: {r['n_inliers']}")
    print(f"  Estimated R:\n{R_est}")
    print(f"  Estimated t: {t_est}")
    print(f"  Expected t direction: [1, 0, 0] (rectified stereo)")
    print(f"  Rotation error (Frobenius): {r['rot_err']:.4f}")
    print(f"  Translation direction error: {r['t_err']:.1f}°")
    print(f"  Triangulated 3D points: {r.get('n_pts3d_valid', 0)} valid")
    if len(sift_pts3d) > 0:
        print(f"  Point depth range: [{sift_pts3d[:,2].min():.0f}, {sift_pts3d[:,2].max():.0f}]")
else:
    print(f"\nSIFT baseline check failed (subprocess exit {proc.returncode}).")
    print(f"  This is a non-essential verification step.")
    if proc.stderr:
        print(f"  stderr: {proc.stderr[:200]}")
    R_est = np.eye(3)
    t_est = np.array([1.0, 0.0, 0.0])

## 7. Triangulate 3D Line Segments

We triangulate each matched segment pair into 3D using three methods:

| Method | Approach | Theory |
|--------|----------|--------|
| `Stereo` | Ray–ray intersection — midpoint of closest approach | Fast, assumes near-rectified |
| `StereoPlane` | Interpretation-plane intersection | Robust for non-rectified pairs |
| `StereoCV` | OpenCV `cv::triangulatePoints` wrapper | Standard DLT baseline |

Each method takes the matched 2D segments and the calibrated camera pair,
and outputs 3D line segments in Plücker representation
$\mathbf{L} = (\mathbf{d}, \mathbf{m})$ where $\mathbf{d}$ is the direction
and $\mathbf{m} = \mathbf{p} \times \mathbf{d}$ is the moment.

We filter out degenerate segments (NaN, infinite, or very large coordinates)
and those with high reprojection error exceeding threshold $\epsilon$.

In [ ]:
t0 = time.perf_counter()

# Build matched segment lists
left_matched = [seg_left[i] for i, _ in matches]
right_matched = [seg_right[m.match_idx] for _, m in matches]


# Convert float segments to double precision for triangulation
def _segments_to_f64(segments):
    """Convert float LineSegment to LineSegment_f64 for stereo triangulation."""
    return [le_geometry.LineSegment_f64.from_endpoints(*s.end_points()) for s in segments]


left_matched_f64 = _segments_to_f64(left_matched)
right_matched_f64 = _segments_to_f64(right_matched)

# Create triangulators (double precision to match Camera_f64)
tri_ray = le_geometry.Stereo_f64(cam_left, cam_right)
tri_plane = le_geometry.StereoPlane_f64(cam_left, cam_right)
tri_cv = le_geometry.StereoCV_f64(cam_left, cam_right)

# Triangulate all matched segments
segs3d_ray = tri_ray.triangulate_segments(left_matched_f64, right_matched_f64)
segs3d_plane = tri_plane.triangulate_segments(left_matched_f64, right_matched_f64)
segs3d_cv = tri_cv.triangulate_segments(left_matched_f64, right_matched_f64)

t_tri = time.perf_counter() - t0


def is_valid_seg3d(seg3d):
    """Check if a 3D segment is valid (non-degenerate, positive depth)."""
    sp = seg3d.start_point()
    ep = seg3d.end_point()
    for v in [*sp, *ep]:
        if np.isnan(v) or np.isinf(v) or abs(v) > 1e6:
            return False
    # Must have positive depth (in front of camera)
    if sp[2] <= 0 or ep[2] <= 0:
        return False
    return seg3d.length > 1e-6


valid_ray = sum(1 for s in segs3d_ray if is_valid_seg3d(s))
valid_plane = sum(1 for s in segs3d_plane if is_valid_seg3d(s))
valid_cv = sum(1 for s in segs3d_cv if is_valid_seg3d(s))

print(f"Triangulation ({t_tri*1000:.1f} ms, {len(matches)} pairs):")
print(f"  Stereo (ray):   {valid_ray}/{len(segs3d_ray)} valid")
print(f"  StereoPlane:    {valid_plane}/{len(segs3d_plane)} valid")
print(f"  StereoCV:       {valid_cv}/{len(segs3d_cv)} valid")

# Build valid segment mask (disparity + depth already filtered upstream)
valid_mask = np.array([is_valid_seg3d(s) for s in segs3d_plane])
n_valid = int(valid_mask.sum())

# Compute endpoint reprojection error (informational only)
cam_proj_left = le_geometry.CameraPluecker_f64(
    focal_x=fx, focal_y=fy, offset_x=cx, offset_y=cy,
    width=calib.get("width", 0), height=calib.get("height", 0),
)
cam_proj_right = le_geometry.CameraPluecker_f64(
    focal_x=fx, focal_y=fy, offset_x=cx_right, offset_y=cy,
    width=calib.get("width", 0), height=calib.get("height", 0),
    tx=baseline,
)


def compute_reproj_error(seg3d, seg2d_left, seg2d_right):
    """Compute endpoint reprojection error for a 3D segment.

    Projects the 3D midpoint back to both views and measures the distance
    to the closest point on the original 2D segment.
    """
    if not is_valid_seg3d(seg3d):
        return np.nan
    try:
        # Project 3D segment midpoint
        sp = np.array(seg3d.start_point())
        ep = np.array(seg3d.end_point())
        mid3d = (sp + ep) / 2

        # Project to both views
        pl = cam_proj_left.project_point(mid3d[0], mid3d[1], mid3d[2])
        pr = cam_proj_right.project_point(mid3d[0], mid3d[1], mid3d[2])

        # Distance to original segment midpoint
        ocl = seg2d_left.center_point()
        ocr = seg2d_right.center_point()
        err_l = np.sqrt((pl[0] - ocl[0]) ** 2 + (pl[1] - ocl[1]) ** 2)
        err_r = np.sqrt((pr[0] - ocr[0]) ** 2 + (pr[1] - ocr[1]) ** 2)
        return (err_l + err_r) / 2.0
    except Exception:
        return np.nan


reproj_errors = np.array([
    compute_reproj_error(s3, sl, sr)
    for s3, sl, sr in zip(segs3d_plane, left_matched, right_matched)
])

valid_errs = reproj_errors[valid_mask & ~np.isnan(reproj_errors)]
print(f"\n3D Validity Filtering:")
print(f"  Disparity-matched pairs: {len(matches)}")
print(f"  Valid 3D segments:       {n_valid} (positive depth, finite)")
if len(valid_errs) > 0:
    print(f"  Midpoint reproj error:   mean={valid_errs.mean():.1f}px, "
          f"med={np.median(valid_errs):.1f}px, max={valid_errs.max():.1f}px")

# Depth statistics
depths = []
for i, v in enumerate(valid_mask):
    if v:
        sp = segs3d_plane[i].start_point()
        ep = segs3d_plane[i].end_point()
        depths.extend([sp[2], ep[2]])
if depths:
    depths = np.array(depths)
    print(f"  Depth range: [{depths.min():.0f}, {depths.max():.0f}], "
          f"median={np.median(depths):.0f}")

## 8. Merge and Cluster Colinear 3D Lines

Post-process triangulated 3D segments by merging near-colinear ones.
Two segments are merged if:
- Angular deviation between direction vectors: $\theta < \theta_{\max}$
- Perpendicular distance between them: $d < d_{\max}$

This reduces redundancy from over-segmented 2D detections and produces
cleaner reconstructions.

In [ ]:
# Merge near-colinear 3D segments using a greedy strategy.
# Two segments are merged if:
#   1. Angular deviation between directions < MERGE_ANGLE_TH
#   2. Perpendicular distance < MERGE_DIST_TH

def seg3d_direction(seg3d):
    """Get unit direction vector of a 3D segment."""
    sp = np.array(seg3d.start_point())
    ep = np.array(seg3d.end_point())
    d = ep - sp
    n = np.linalg.norm(d)
    return d / n if n > 1e-12 else np.array([0, 0, 1])

def seg3d_midpoint(seg3d):
    """Get midpoint of a 3D segment."""
    sp = np.array(seg3d.start_point())
    ep = np.array(seg3d.end_point())
    return (sp + ep) / 2.0

def point_line_distance(point, line_origin, line_dir):
    """Perpendicular distance from a point to an infinite line."""
    v = point - line_origin
    proj = np.dot(v, line_dir) * line_dir
    return float(np.linalg.norm(v - proj))

def can_merge(seg_a, seg_b, angle_th_deg, dist_th):
    """Check if two 3D segments are close enough to merge."""
    da = seg3d_direction(seg_a)
    db = seg3d_direction(seg_b)
    cos_angle = abs(np.dot(da, db))
    cos_angle = min(cos_angle, 1.0)
    angle_deg = np.degrees(np.arccos(cos_angle))
    if angle_deg > angle_th_deg:
        return False
    mid_b = seg3d_midpoint(seg_b)
    mid_a = seg3d_midpoint(seg_a)
    dist = point_line_distance(mid_b, mid_a, da)
    return dist < dist_th

def depth_axis_angle(seg3d):
    """Angle (degrees) between segment direction and the Z (depth) axis.

    Returns 0° when perfectly aligned with depth, 90° when perpendicular.
    """
    d = seg3d_direction(seg3d)
    return np.degrees(np.arccos(np.clip(abs(d[2]), 0, 1)))


def is_depth_outlier(seg3d, hard_angle, soft_angle, length_sat):
    """Adaptive outlier test: longer segments get more tolerance.

    The Z-angle threshold interpolates linearly between soft_angle (for very
    short segments) and hard_angle (for segments longer than length_sat).
    This reflects that short depth-parallel segments are likely triangulation
    artifacts, while long ones *might* be real but are still suspicious.

    Returns True if the segment should be REJECTED.
    """
    z_angle = depth_axis_angle(seg3d)  # 0° = aligned with Z, 90° = perpendicular
    seg_len = seg3d.length

    # Always reject if below hard limit
    if z_angle < hard_angle:
        return True

    # Interpolate threshold: short segments need z_angle >= soft_angle,
    # long segments only need z_angle >= hard_angle
    t = np.clip(seg_len / length_sat, 0, 1)
    threshold = soft_angle * (1 - t) + hard_angle * t

    return z_angle < threshold


# Filter to valid segments first
valid_indices = [i for i in range(len(segs3d_plane))
                 if is_valid_seg3d(segs3d_plane[i]) and valid_mask[i]]
valid_segs = [segs3d_plane[i] for i in valid_indices]
valid_left = [left_matched[i] for i in valid_indices]
valid_right = [right_matched[i] for i in valid_indices]
valid_errors = reproj_errors[valid_indices]

# Greedy merging
n_valid = len(valid_segs)
cluster_id = list(range(n_valid))
merged = [False] * n_valid

for i in range(n_valid):
    if merged[i]:
        continue
    for j in range(i + 1, n_valid):
        if merged[j]:
            continue
        if can_merge(valid_segs[i], valid_segs[j], MERGE_ANGLE_TH, MERGE_DIST_TH):
            cluster_id[j] = cluster_id[i]
            merged[j] = True

from collections import defaultdict
clusters = defaultdict(list)
for i, cid in enumerate(cluster_id):
    root = cid
    while cluster_id[root] != root:
        root = cluster_id[root]
    clusters[root].append(i)

merged_segs = []
merged_left = []
merged_right = []
merged_errors = []
for root, members in clusters.items():
    best = max(members, key=lambda i: valid_segs[i].length)
    merged_segs.append(valid_segs[best])
    merged_left.append(valid_left[best])
    merged_right.append(valid_right[best])
    merged_errors.append(valid_errors[best])

merged_errors = np.array(merged_errors)
n_after_merge = len(merged_segs)

print(f"Colinear Merging (angle < {MERGE_ANGLE_TH}°, dist < {MERGE_DIST_TH}):")
print(f"  Before: {n_valid} valid segments")
print(f"  After:  {n_after_merge} merged segments")
print(f"  Reduction: {n_valid - n_after_merge} removed "
      f"({(1 - n_after_merge/max(n_valid,1))*100:.0f}%)")

# --- Adaptive outlier detection: angle + length ---
# Short segments aligned with Z are almost certainly triangulation artifacts.
# Longer segments get more tolerance (the threshold relaxes with length).
outlier_flags = np.array([
    is_depth_outlier(s, OUTLIER_Z_ANGLE_HARD, OUTLIER_Z_ANGLE_SOFT, OUTLIER_LENGTH_SAT)
    for s in merged_segs
])
n_outliers = int(outlier_flags.sum())

# Collect stats before filtering
z_angles = np.array([depth_axis_angle(s) for s in merged_segs])
lengths_3d = np.array([s.length for s in merged_segs])

if n_outliers > 0:
    kept = [(s, l, r, e) for s, l, r, e, reject
            in zip(merged_segs, merged_left, merged_right, merged_errors, outlier_flags)
            if not reject]
    merged_segs = [x[0] for x in kept]
    merged_left = [x[1] for x in kept]
    merged_right = [x[2] for x in kept]
    merged_errors = np.array([x[3] for x in kept])

print(f"\nOutlier Detection (adaptive angle + length):")
print(f"  Hard Z-angle limit: {OUTLIER_Z_ANGLE_HARD}° (always reject)")
print(f"  Soft Z-angle limit: {OUTLIER_Z_ANGLE_SOFT}° (for short segments)")
print(f"  Length saturation:  {OUTLIER_LENGTH_SAT} world units")
print(f"  Rejected: {n_outliers}, Kept: {len(merged_segs)}")
if n_outliers > 0:
    rej_angles = z_angles[outlier_flags]
    rej_lens = lengths_3d[outlier_flags]
    print(f"  Rejected — Z-angles: [{rej_angles.min():.1f}°, {rej_angles.max():.1f}°], "
          f"lengths: [{rej_lens.min():.0f}, {rej_lens.max():.0f}]")
kept_angles = z_angles[~outlier_flags]
if len(kept_angles) > 0:
    kept_lens = lengths_3d[~outlier_flags]
    print(f"  Kept     — Z-angles: [{kept_angles.min():.1f}°, {np.median(kept_angles):.1f}° med], "
          f"lengths: [{kept_lens.min():.0f}, {np.median(kept_lens):.0f} med]")

## 9. Visualize 3D Line Reconstruction with Rerun

We log the full reconstruction to [Rerun](https://rerun.io):
- **Camera frustums** with images projected onto them
- **3D line segments** color-coded by depth (blue = close, red = far)
- **SIFT point cloud** for scene context
- **2D overlays** showing detected and matched segments

Use the **timeline scrubber** (bottom bar) to switch views:
- `step=0` — All detected segments in both images
- `step=1` — Stereo-matched segments (color = depth)
- `step=2` — 3D line reconstruction with camera frustums
- `step=3` — 3D lines + SIFT point cloud

In [ ]:
rr.init("stereo_reconstruction")

def generate_colors(n):
    """Generate N visually distinct colors via HSV spacing."""
    colors = []
    for i in range(n):
        h = i / max(n, 1)
        r, g, b = colorsys.hsv_to_rgb(h, 0.85, 0.95)
        colors.append([int(r * 255), int(g * 255), int(b * 255)])
    return np.array(colors, dtype=np.uint8)

def seg_to_strip(seg):
    """Convert a 2D LineSegment to [[x1,y1],[x2,y2]]."""
    x1, y1, x2, y2 = seg.end_points()
    return [[x1, y1], [x2, y2]]

def seg3d_to_strip(seg3d):
    """Convert a 3D LineSegment3 to [[x,y,z],[x,y,z]]."""
    sp = seg3d.start_point()
    ep = seg3d.end_point()
    return [[sp[0], sp[1], sp[2]], [ep[0], ep[1], ep[2]]]

def depth_to_color(seg3d, min_d=None, max_d=None):
    """Map average depth to blue (close) → green → red (far)."""
    sp = seg3d.start_point()
    ep = seg3d.end_point()
    d = (sp[2] + ep[2]) / 2.0
    if min_d is None or max_d is None or max_d <= min_d:
        return [0, 200, 0]
    t = np.clip((d - min_d) / (max_d - min_d), 0, 1)
    # Blue → Cyan → Green → Yellow → Red
    if t < 0.25:
        s = t / 0.25
        return [0, int(s * 255), 255]
    elif t < 0.5:
        s = (t - 0.25) / 0.25
        return [0, 255, int((1 - s) * 255)]
    elif t < 0.75:
        s = (t - 0.5) / 0.25
        return [int(s * 255), 255, 0]
    else:
        s = (t - 0.75) / 0.25
        return [255, int((1 - s) * 255), 0]

_OVERLAY_ENTITIES = [
    "left/segments", "right/segments",
    "left/matches", "right/matches",
]

def clear_overlays():
    """Clear segment overlays at the current timestep."""
    for e in _OVERLAY_ENTITIES:
        rr.log(e, rr.Clear(recursive=False))

In [ ]:
# --- Step 0: All detected segments ---
rr.set_time("step", sequence=0)
clear_overlays()

rr.log("left/image", rr.Image(img_left_color))
left_strips = [seg_to_strip(s) for s in seg_left]
left_colors = generate_colors(len(seg_left))
rr.log("left/segments", rr.LineStrips2D(left_strips, colors=left_colors, radii=1.0))
rr.log("left/info", rr.TextLog(
    f"Detected {len(seg_left)} segments (>= {MIN_SEG_LENGTH}px)"))

rr.log("right/image", rr.Image(img_right_color))
right_strips = [seg_to_strip(s) for s in seg_right]
right_colors = generate_colors(len(seg_right))
rr.log("right/segments", rr.LineStrips2D(right_strips, colors=right_colors, radii=1.0))
rr.log("right/info", rr.TextLog(
    f"Detected {len(seg_right)} segments (>= {MIN_SEG_LENGTH}px)"))

print(f"Step 0: {len(seg_left)} left + {len(seg_right)} right segments")

In [ ]:
# --- Step 1: Matched segments, color-coded by depth ---
rr.set_time("step", sequence=1)
clear_overlays()

n_show = min(N_SHOW, len(merged_segs))

# Compute depth range for color mapping
all_depths = []
for seg3 in merged_segs[:n_show]:
    sp = seg3.start_point()
    ep = seg3.end_point()
    all_depths.extend([sp[2], ep[2]])
min_d = min(all_depths) if all_depths else 0
max_d = max(all_depths) if all_depths else 1

show_colors = np.array(
    [depth_to_color(merged_segs[i], min_d, max_d) for i in range(n_show)],
    dtype=np.uint8)

rr.log("left/image", rr.Image(img_left_color))
rr.log("right/image", rr.Image(img_right_color))

left_match_strips = [seg_to_strip(merged_left[i]) for i in range(n_show)]
right_match_strips = [seg_to_strip(merged_right[i]) for i in range(n_show)]
rr.log("left/matches", rr.LineStrips2D(
    left_match_strips, colors=show_colors, radii=2.0))
rr.log("right/matches", rr.LineStrips2D(
    right_match_strips, colors=show_colors, radii=2.0))

rr.log("logs", rr.TextLog(
    f"Matched segments: {n_show} pairs  "
    f"(blue=close, green=mid, red=far, depth range [{min_d:.0f}, {max_d:.0f}])"))

print(f"Step 1: {n_show} matched pairs (color = depth)")

In [ ]:
# --- Step 2: 3D reconstruction + camera frustums ---
rr.set_time("step", sequence=2)

rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

# Compute depth cutoff (90th percentile) to exclude extreme outliers
all_seg_depths = []
for seg3 in merged_segs:
    if is_valid_seg3d(seg3):
        sp = seg3.start_point()
        ep = seg3.end_point()
        all_seg_depths.append((sp[2] + ep[2]) / 2.0)
if all_seg_depths:
    depth_cutoff = np.percentile(all_seg_depths, 90)
    median_depth = np.median(all_seg_depths)
else:
    depth_cutoff = 1e6
    median_depth = 1000

# Line radius proportional to scene scale (visible at any zoom level)
line_radius = max(median_depth * 0.003, 1.0)

# Log merged 3D line segments (color by depth, filtered by cutoff)
strips_3d = []
colors_3d = []
n_clipped = 0
for seg3 in merged_segs:
    if not is_valid_seg3d(seg3):
        continue
    sp = seg3.start_point()
    ep = seg3.end_point()
    avg_d = (sp[2] + ep[2]) / 2.0
    if avg_d > depth_cutoff:
        n_clipped += 1
        continue
    strips_3d.append(seg3d_to_strip(seg3))
    colors_3d.append(depth_to_color(seg3, min_d, depth_cutoff))

if strips_3d:
    rr.log("world/lines", rr.LineStrips3D(
        strips_3d, colors=colors_3d, radii=line_radius))

# Camera frustums with images
w_img = calib.get("width", img_left.shape[1])
h_img = calib.get("height", img_left.shape[0])

# Left camera at origin
rr.log("world/cam_left", rr.Pinhole(
    focal_length=[fx, fy],
    principal_point=[cx, cy],
    resolution=[w_img, h_img],
))
rr.log("world/cam_left", rr.Transform3D(translation=[0.0, 0.0, 0.0]))
rr.log("world/cam_left/image", rr.Image(img_left_color))

# Right camera shifted by baseline
rr.log("world/cam_right", rr.Pinhole(
    focal_length=[fx, fy],
    principal_point=[cx_right, cy],
    resolution=[w_img, h_img],
))
rr.log("world/cam_right", rr.Transform3D(
    translation=[baseline, 0.0, 0.0]))
rr.log("world/cam_right/image", rr.Image(img_right_color))

rr.log("logs", rr.TextLog(
    f"3D reconstruction: {len(strips_3d)} segments "
    f"(depth cutoff {depth_cutoff:.0f}, clipped {n_clipped}), "
    f"line_radius={line_radius:.1f}"))

print(f"Step 2: {len(strips_3d)} 3D segments + 2 camera frustums "
      f"(clipped {n_clipped} beyond depth {depth_cutoff:.0f}, "
      f"radius={line_radius:.1f})")

In [ ]:
# --- Step 3: 3D lines + SIFT point cloud ---
rr.set_time("step", sequence=3)

if len(sift_pts3d) > 0:
    # Apply same depth cutoff as lines for consistency
    pt_depths = sift_pts3d[:, 2]
    pt_mask = (pt_depths > 0) & (pt_depths <= depth_cutoff)
    pts_vis = sift_pts3d[pt_mask]

    # Color points by depth (same colormap as lines)
    pt_colors = []
    for p in pts_vis:
        d_val = p[2]
        t = np.clip((d_val - min_d) / (depth_cutoff - min_d + 1e-6), 0, 1)
        if t < 0.25:
            s = t / 0.25
            pt_colors.append([0, int(s * 255), 255])
        elif t < 0.5:
            s = (t - 0.25) / 0.25
            pt_colors.append([0, 255, int((1 - s) * 255)])
        elif t < 0.75:
            s = (t - 0.5) / 0.25
            pt_colors.append([int(s * 255), 255, 0])
        else:
            s = (t - 0.75) / 0.25
            pt_colors.append([255, int((1 - s) * 255), 0])

    pt_radius = max(line_radius * 0.4, 1.0)
    rr.log("world/sift_points", rr.Points3D(
        pts_vis, colors=pt_colors, radii=pt_radius))

    print(f"Step 3: {len(pts_vis)} SIFT 3D points + {len(strips_3d)} line segments "
          f"(depth cutoff {depth_cutoff:.0f}, point radius={pt_radius:.1f})")
else:
    print("Step 3: No SIFT 3D points available (SIFT subprocess may have failed)")

In [ ]:
# Render inline with explicit layout:
# Top: Left image | Right image
# Bottom: 3D reconstruction (larger)
# Use timeline scrubber: step 0=detections, 1=matches, 2=3D lines, 3=lines+points
blueprint = rrb.Blueprint(
    rrb.Vertical(
        rrb.Horizontal(
            rrb.Spatial2DView(name="Left Image", origin="left"),
            rrb.Spatial2DView(name="Right Image", origin="right"),
        ),
        rrb.Spatial3DView(name="3D Reconstruction", origin="world"),
        row_shares=[1, 2],
    ),
    collapse_panels=True,
)
rr.notebook_show(width=1200, height=900, blueprint=blueprint)

## 10. Evaluate Reconstruction Quality

Quantitative evaluation of the reconstruction:
- **Reprojection error** — per-line and aggregate (mean/median/max)
- **Completeness** — fraction of 2D lines successfully triangulated
- **Method comparison** — all three triangulation methods side by side
- **Timing breakdown** — per-stage performance summary

In [ ]:
def depth_stats(segs3d):
    """Compute depth (Z) statistics for valid segments."""
    depths = []
    for seg3 in segs3d:
        if is_valid_seg3d(seg3):
            sp = seg3.start_point()
            ep = seg3.end_point()
            depths.extend([sp[2], ep[2]])
    return np.array(depths) if depths else np.array([])

def length_stats(segs3d):
    """Compute 3D segment length statistics."""
    lengths = [seg3.length for seg3 in segs3d if is_valid_seg3d(seg3)]
    return np.array(lengths) if lengths else np.array([])

# Compute reprojection errors for all three methods
err_ray_l, err_ray_r = [], []
err_plane_l, err_plane_r = [], []
err_cv_l, err_cv_r = [], []
for i in range(len(left_matched)):
    for segs3d, el, er in [
        (segs3d_ray, err_ray_l, err_ray_r),
        (segs3d_plane, err_plane_l, err_plane_r),
        (segs3d_cv, err_cv_l, err_cv_r),
    ]:
        e = compute_reproj_error(
            segs3d[i], left_matched[i], right_matched[i])
        el.append(e)
        er.append(e)  # symmetric for this metric
err_ray_l = np.array(err_ray_l)
err_plane_l = np.array(err_plane_l)
err_cv_l = np.array(err_cv_l)

print(f"{'='*80}")
print(f"Stereo Line Reconstruction — Evaluation Summary")
print(f"{'='*80}")
print(f"Scene: {SCENE} ({RESOLUTION}), Image size: {img_left.shape[1]}x{img_left.shape[0]}")
print()

# Completeness
total_left = len(seg_left)
total_right = len(seg_right)
n_matched = len(matches)
n_triangulated = sum(1 for s in segs3d_plane if is_valid_seg3d(s))
n_after_valid = int(valid_mask.sum())
n_after_merge = len(merged_segs)

print(f"Pipeline Completeness:")
print(f"  Detected segments:       {total_left} left, {total_right} right")
print(f"  Stereo-matched pairs:    {n_matched} ({n_matched/min(total_left,total_right)*100:.1f}%)")
print(f"  Valid triangulations:    {n_triangulated} ({n_triangulated/max(n_matched,1)*100:.1f}%)")
print(f"  Valid 3D (depth+finite): {n_after_valid}")
print(f"  After colinear merge:    {n_after_merge}")
print()

# Method comparison
print(f"Method Comparison (midpoint reprojection error, pixels):")
header = f"  {'Method':<16s}  {'Valid':>5s}  {'Mean':>8s}  {'Median':>8s}  {'Std':>8s}  {'Max':>8s}  {'<5px':>5s}"
print(header)
print(f"  {'-'*66}")

for name, segs3d, errs in [
    ("Stereo (ray)", segs3d_ray, err_ray_l),
    ("StereoPlane", segs3d_plane, err_plane_l),
    ("StereoCV", segs3d_cv, err_cv_l),
]:
    valid = sum(1 for s in segs3d if is_valid_seg3d(s))
    v = errs[~np.isnan(errs)]
    if len(v) > 0:
        n_under5 = int((v < 5.0).sum())
        print(f"  {name:<16s}  {valid:>5d}  {np.mean(v):>6.2f}px"
              f"  {np.median(v):>6.2f}px  {np.std(v):>6.2f}px"
              f"  {np.max(v):>6.1f}px  {n_under5:>5d}")
    else:
        print(f"  {name:<16s}  {valid:>5d}  {'N/A':>8s}  {'N/A':>8s}"
              f"  {'N/A':>8s}  {'N/A':>8s}  {'N/A':>5s}")
print()

# 3D reconstruction statistics
d = depth_stats(merged_segs)
l = length_stats(merged_segs)
if len(d) > 0:
    print(f"3D Reconstruction Statistics (merged segments):")
    print(f"  Depth (Z):   min={d.min():.1f}, median={np.median(d):.1f}, "
          f"max={d.max():.1f}")
    print(f"  Length (3D):  min={l.min():.3f}, median={np.median(l):.3f}, "
          f"max={l.max():.3f}")
    print()

# Timing
print(f"Timing Breakdown:")
print(f"  Detection:       {t_detect*1000:>8.1f} ms")
print(f"  Descriptors:     {t_desc*1000:>8.1f} ms")
print(f"  Raw matching:    {t_raw_match*1000:>8.1f} ms")
print(f"  Stereo filter:   {t_filter*1000:>8.1f} ms")
print(f"  Triangulation:   {t_tri*1000:>8.1f} ms")
t_total = t_detect + t_desc + t_raw_match + t_filter + t_tri
print(f"  {'Total:':<17s} {t_total*1000:>8.1f} ms")
print()

# Error distribution histogram (text-based)
if len(merged_errors) > 0:
    bins = [0, 1, 2, 5, 10, 20, 50, 100, np.inf]
    counts, _ = np.histogram(merged_errors[~np.isnan(merged_errors)], bins=bins)
    print(f"Midpoint Reproj Error Distribution (merged, {len(merged_errors)} segs):")
    for i in range(len(counts)):
        lo = bins[i]
        hi = bins[i+1]
        bar = "#" * int(counts[i] * 40 / max(counts.max(), 1))
        label = f"  [{lo:>5.0f}, {hi:>5.0f})" if hi < np.inf else f"  [{lo:>5.0f},   inf)"
        print(f"{label}  {counts[i]:>4d}  {bar}")

## Summary

This demo showed the complete stereo line reconstruction pipeline:

| Stage | Component | Notes |
|-------|-----------|-------|
| Detection | `LsdCC` | Connected-component LSD, native Sobel gradients |
| Descriptors | `FdcLBD` | 72-dim float descriptors, L2 distance |
| Raw Matching | `BruteForceLBD` | Ratio test filtering |
| Geometric Filter | `StereoLineFilter` + `GlobalRotationFilter` | Epipolar + rotation |
| Triangulation | `Stereo` / `StereoPlane` / `StereoCV` | Three methods compared |
| Merging | Greedy colinear merge | Angle + distance threshold |
| Evaluation | Reprojection error | 3D → 2D midpoint distance |
| Visualization | Rerun | 2D overlays + 3D scene + camera frustums |

### Key Observations

- **StereoPlane** works best for general (potentially non-rectified) stereo pairs
- **Stereo** (ray intersection) is fastest and works well for rectified data
- **StereoCV** provides an OpenCV-compatible baseline for comparison
- The `StereoLineFilter` dramatically reduces false matches by enforcing
  epipolar geometry before descriptor comparison
- Colinear merging reduces segment count while preserving scene structure

### Next Steps

- **Parameter tuning** — experiment with detection and matching thresholds
- **Point + line fusion** — combine point and line features for robust SfM